In [10]:
import os
import glob
import numpy as np
import rasterio
from rasterio.enums import Resampling
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
import seaborn as sns
from pathlib import Path
from scipy.interpolate import interp1d
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (confusion_matrix, classification_report,
                             f1_score, accuracy_score)
import warnings
warnings.filterwarnings("ignore")
 

In [12]:
class Config:
    # ── Paths ──────────────────────────────────────────────────────────────
    SAFE_DIR   = r"C:\SII\aarn\projet"          # folder containing .SAFE directories
    CDL_PATH   = r"C:\SII\aarn\projet\cdl\cdl_california_2021.tif"  # CDL GeoTIFF
    OUTPUT_DIR = r"C:\SII\aarn\projet\outputs"
 
    # ── Sentinel-2 bands to use (10 bands as in the paper) ─────────────────
    BANDS = ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12"]
    RESOLUTION = "10m"    # "10m", "20m", or "60m"
 
    # ── Temporal settings ──────────────────────────────────────────────────
    YEAR       = 2021
    N_TIMESTEPS = 30       # number of temporal samples (paper uses ~30)
    CLOUD_THRESH = 0.2     # max cloud fraction per scene
 
    # ── CDL crop classes to keep (USDA codes) ─────────────────────────────
    CDL_CLASSES = {
        1:  "Corn",
        2:  "Cotton",
        5:  "Soybeans",
        6:  "Sunflower",
        21: "Wheat",
        24: "Winter Wheat",
        75: "Almonds",
        204:"Pistachios",
        3:  "Rice",
    }
 
    # ── Model hyper-parameters ─────────────────────────────────────────────
    PATCH_SIZE  = 1        # pixel-based (no spatial context)
    N_BANDS     = 10
    N_CLASSES   = len(CDL_CLASSES)
    HIDDEN_DIM  = 64
    N_HEADS     = 4
    N_LAYERS    = 2
    DROPOUT     = 0.1
    BATCH_SIZE  = 512
    EPOCHS      = 50
    LR          = 1e-3
    TRAIN_RATIO = 0.7
    VAL_RATIO   = 0.15
 
os.makedirs(Config.OUTPUT_DIR, exist_ok=True)

In [13]:
def find_safe_scenes(safe_dir: str):
    """Return sorted list of .SAFE directories."""
    scenes = sorted(glob.glob(os.path.join(safe_dir, "*.SAFE")))
    print(f"[INFO] Found {len(scenes)} .SAFE scenes in {safe_dir}")
    return scenes
 
 
def parse_scene_date(safe_path: str):
    """Extract acquisition date from .SAFE folder name."""
    name = Path(safe_path).name          # e.g. S2A_MSIL2A_20210628T184921_...
    date_str = name.split("_")[2][:8]   # '20210628'
    return np.datetime64(f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}")
 
 
def read_scene_bands(safe_path: str, bands: list, resolution: str = "10m"):
    """
    Read selected bands from a .SAFE directory.
    Returns dict {band_name: 2D np.array} and the rasterio profile.
    """
    res_folder_map = {"10m": "R10m", "20m": "R20m", "60m": "R60m"}
    res_folder = res_folder_map[resolution]
 
    granule_dir = glob.glob(
        os.path.join(safe_path, "GRANULE", "*", "IMG_DATA", res_folder)
    )
    if not granule_dir:
        # try any resolution as fallback
        granule_dir = glob.glob(
            os.path.join(safe_path, "GRANULE", "*", "IMG_DATA", "R*m")
        )
    if not granule_dir:
        raise FileNotFoundError(f"No IMG_DATA found in {safe_path}")
 
    granule_dir = granule_dir[0]
    band_data = {}
    profile = None
 
    for band in bands:
        jp2_files = glob.glob(os.path.join(granule_dir, f"*_{band}_*.jp2"))
        if not jp2_files:
            # try without resolution suffix in filename
            jp2_files = glob.glob(os.path.join(granule_dir, f"*{band}*.jp2"))
        if not jp2_files:
            print(f"  [WARN] Band {band} not found in {granule_dir}")
            continue
 
        with rasterio.open(jp2_files[0]) as src:
            band_data[band] = src.read(1).astype(np.float32)
            if profile is None:
                profile = src.profile.copy()
 
    return band_data, profile
 
 
def read_scl_mask(safe_path: str):
    """Read Scene Classification Layer (cloud mask) at 20m."""
    scl_files = glob.glob(
        os.path.join(safe_path, "GRANULE", "*", "IMG_DATA", "R20m", "*SCL_20m.jp2")
    )
    if not scl_files:
        scl_files = glob.glob(
            os.path.join(safe_path, "GRANULE", "*", "IMG_DATA", "**", "*SCL*.jp2"),
            recursive=True
        )
    if not scl_files:
        return None
    with rasterio.open(scl_files[0]) as src:
        return src.read(1)
 
 
def get_cloud_fraction(scl_array):
    """Compute cloud fraction from SCL (classes 8,9,10 = clouds)."""
    if scl_array is None:
        return 0.0
    cloud_mask = np.isin(scl_array, [8, 9, 10])
    return cloud_mask.mean()
 
 
# ─────────────────────────────────────────────────────────────────────────────
# 3. VEGETATION INDICES
# ─────────────────────────────────────────────────────────────────────────────
def compute_indices(bands: dict):
    """Compute NDVI, EVI, NDWI, LSWI from band dictionary."""
    eps = 1e-8
    # reflectance: divide by 10000 for L2A
    B04 = bands.get("B04", None)
    B08 = bands.get("B08", None)
    B03 = bands.get("B03", None)
    B11 = bands.get("B11", None)
    B02 = bands.get("B02", None)
 
    indices = {}
 
    if B04 is not None and B08 is not None:
        nir, red = B08 / 10000.0, B04 / 10000.0
        indices["NDVI"] = (nir - red) / (nir + red + eps)
 
        if B02 is not None:
            blue = B02 / 10000.0
            indices["EVI"] = 2.5 * (nir - red) / (nir + 6*red - 7.5*blue + 1 + eps)
 
    if B03 is not None and B11 is not None:
        green = B03 / 10000.0
        swir  = B11 / 10000.0
        if B08 is not None:
            indices["NDWI"]  = (green - nir) / (green + nir + eps)
            indices["LSWI"]  = (nir - swir)  / (nir + swir + eps)
 
    return indices

In [14]:
def build_time_series(scenes: list, bands: list, resolution: str,
                      cloud_thresh: float = 0.2):
    """
    Load all scenes, filter cloudy ones, return:
        ts_data  : list of dicts {band: 2D array, date: datetime64}
        ts_indices: list of dicts {index_name: 2D array, date: datetime64}
    """
    ts_data    = []
    ts_indices = []
    dates      = []
 
    print("\n[INFO] Loading scenes …")
    for safe in scenes:
        date = parse_scene_date(safe)
        scl  = read_scl_mask(safe)
        cf   = get_cloud_fraction(scl)
 
        if cf > cloud_thresh:
            print(f"  SKIP {Path(safe).name}  cloud={cf:.1%}")
            continue
 
        try:
            band_data, _ = read_scene_bands(safe, bands, resolution)
        except FileNotFoundError as e:
            print(f"  SKIP {Path(safe).name}  {e}")
            continue
 
        indices = compute_indices(band_data)
 
        ts_data.append({"date": date, **band_data})
        ts_indices.append({"date": date, **indices})
        dates.append(date)
        print(f"  OK   {Path(safe).name}  cloud={cf:.1%}")
 
    print(f"\n[INFO] {len(ts_data)} scenes passed cloud filter")
    return ts_data, ts_indices, np.array(dates)

In [15]:
def temporal_interpolate(ts_list: list, dates: np.ndarray,
                         n_steps: int = 30, year: int = 2021):
    """
    Interpolate each band / index to a fixed temporal grid.
    Returns array of shape (n_steps, H, W, n_channels).
    """
    if len(ts_list) == 0:
        raise ValueError("No scenes available for interpolation")
 
    start = np.datetime64(f"{year}-01-01")
    end   = np.datetime64(f"{year}-12-31")
    target_dates = np.linspace(0, (end - start).astype(int),
                               n_steps, dtype=int)
    src_days = ((dates - start).astype(int))
 
    # get channel names (exclude 'date')
    channels = [k for k in ts_list[0].keys() if k != "date"]
    H, W     = ts_list[0][channels[0]].shape
 
    result   = np.zeros((n_steps, H, W, len(channels)), dtype=np.float32)
 
    for ci, ch in enumerate(channels):
        stack = np.stack([scene[ch] for scene in ts_list], axis=0)  # (T,H,W)
        for r in range(H):
            for c in range(W):
                pixel_ts = stack[:, r, c]
                f = interp1d(src_days, pixel_ts, kind="linear",
                             bounds_error=False,
                             fill_value=(pixel_ts[0], pixel_ts[-1]))
                result[:, r, c, ci] = f(target_dates)
 
    print(f"[INFO] Interpolated to shape {result.shape}")
    return result, channels

In [16]:
def normalize(data: np.ndarray):
    """Per-channel z-score normalization across (T, H, W) dimensions."""
    T, H, W, C = data.shape
    flat = data.reshape(-1, C)
    mean = flat.mean(axis=0)
    std  = flat.std(axis=0) + 1e-8
    return (data - mean) / std, mean, std
 

In [18]:
def plot_rgb(band_data: dict, date, save_path=None):
    """Plot true-color composite (B04=R, B03=G, B02=B)."""
    r = np.clip(band_data["B04"] / 3000.0, 0, 1)
    g = np.clip(band_data["B03"] / 3000.0, 0, 1)
    b = np.clip(band_data["B02"] / 3000.0, 0, 1)
    rgb = np.dstack([r, g, b])
 
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(rgb)
    ax.set_title(f"True Color Composite — {date}", fontsize=13)
    ax.axis("off")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        print(f"[SAVED] {save_path}")
    plt.show()
 
 
def plot_ndvi_timeseries(ts_indices: list, dates: np.ndarray,
                         sample_points=5, save_path=None):
    """
    Plot NDVI time-series for random pixel samples.
    """
    if "NDVI" not in ts_indices[0]:
        print("[WARN] NDVI not available")
        return
 
    ndvi_stack = np.stack([s["NDVI"] for s in ts_indices], axis=0)  # (T,H,W)
    H, W = ndvi_stack.shape[1:]
 
    np.random.seed(42)
    rows = np.random.randint(0, H, sample_points)
    cols = np.random.randint(0, W, sample_points)
 
    fig, ax = plt.subplots(figsize=(12, 5))
    colors = plt.cm.Set1(np.linspace(0, 1, sample_points))
 
    for i, (r, c) in enumerate(zip(rows, cols)):
        ts = ndvi_stack[:, r, c]
        ax.plot(dates, ts, "o-", color=colors[i], label=f"Pixel ({r},{c})", lw=2)
 
    ax.axhline(0, color="k", lw=0.5, ls="--")
    ax.set_xlabel("Date")
    ax.set_ylabel("NDVI")
    ax.set_title("NDVI Time-Series for Sample Pixels")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        print(f"[SAVED] {save_path}")
    plt.show()
 
 
def plot_class_distribution(cdl_array: np.ndarray, cdl_classes: dict,
                            save_path=None):
    """Bar chart of crop class distribution."""
    counts = {}
    for code, name in cdl_classes.items():
        cnt = (cdl_array == code).sum()
        if cnt > 0:
            counts[name] = cnt
 
    if not counts:
        print("[WARN] No CDL classes found in array")
        return
 
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(list(counts.keys()), list(counts.values()),
                  color=plt.cm.tab10(np.linspace(0, 1, len(counts))))
    ax.set_xlabel("Crop Type")
    ax.set_ylabel("Pixel Count")
    ax.set_title("Class Distribution (CDL)")
    ax.bar_label(bars, fmt="%d", fontsize=8)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        print(f"[SAVED] {save_path}")
    plt.show()
 
 
def plot_band_heatmap(ts_data: list, band: str = "B08", save_path=None):
    """Heatmap of mean band values across scenes."""
    stack = np.stack([s[band] / 10000.0 for s in ts_data
                      if band in s], axis=0)
    mean_img = stack.mean(axis=0)
 
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(mean_img, cmap="YlGn")
    plt.colorbar(im, ax=ax, label=f"Mean {band} Reflectance")
    ax.set_title(f"Mean {band} (NIR) — All Scenes")
    ax.axis("off")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        print(f"[SAVED] {save_path}")
    plt.show()
 
 
def plot_temporal_patterns_by_class(ts_indices, dates, cdl_array,
                                    cdl_classes, index="NDVI", save_path=None):
    """Mean NDVI (or other index) temporal profile per crop class."""
    ndvi_stack = np.stack([s[index] for s in ts_indices
                           if index in s], axis=0)  # (T, H, W)
 
    T, H, W = ndvi_stack.shape
    cdl_flat = cdl_array[:H, :W]  # ensure same spatial extent
 
    fig, ax = plt.subplots(figsize=(13, 6))
    colors = plt.cm.tab10(np.linspace(0, 1, len(cdl_classes)))
 
    for (code, name), color in zip(cdl_classes.items(), colors):
        mask = cdl_flat == code
        if mask.sum() < 10:
            continue
        mean_ts = ndvi_stack[:, mask].mean(axis=1)
        ax.plot(dates, mean_ts, "o-", label=name, color=color, lw=2)
 
    ax.set_xlabel("Date")
    ax.set_ylabel(index)
    ax.set_title(f"Mean {index} Temporal Profile by Crop Class")
    ax.legend(bbox_to_anchor=(1.01, 1), fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        print(f"[SAVED] {save_path}")
    plt.show()
 
 